### Sentiment Analysis on SEC EDGAR files

We calculate the sentiments over the parsed raw data we get from 10-K and 10-Q Sec files for past 10 years for aribitrarely picked 100 tickers from S&P500. 

In [4]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import os
import re
import pandas as pd
from nltk.tokenize import RegexpTokenizer, sent_tokenize
import numpy as np
import nltk
nltk.download('punkt')
from nltk import word_tokenize

sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/kiranrawat/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
common_path = '/Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml'

#### Vader api for Sentiment Analysis

VADER belongs to a type of sentiment analysis that is based on lexicons of sentiment-related words. In this approach, each of the words in the lexicon is rated as to whether it is positive or negative, and in many cases, how positive or negative. We update the VADER's sentiment lexicon with stock market lexicon and also modify it's positive and negative dictionaries.

In [7]:
import csv
import pandas as pd

# stock market lexicon
stock_lex = pd.read_csv(common_path+'/EDGAR-reports-Text-Analysis/stock_lex.csv')

stock_lex['sentiment'] = (stock_lex['Aff_Score'] + stock_lex['Neg_Score'])/2
stock_lex = dict(zip(stock_lex.Item, stock_lex.sentiment))
stock_lex = {k:v for k,v in stock_lex.items() if len(k.split(' '))==1}
stock_lex_scaled = {}
for k, v in stock_lex.items():
    if v > 0:
        stock_lex_scaled[k] = v / max(stock_lex.values()) * 4
    else:
        stock_lex_scaled[k] = v / min(stock_lex.values()) * -4

#update the positive and negative dictionaries
positive = []
with open(common_path+'/EDGAR-reports-Text-Analysis/lm_positive.csv', 'r') as f:
    reader = csv.reader(f)
    for row in reader:
        positive.append(row[0].strip())
    
negative = []
with open(common_path+'/EDGAR-reports-Text-Analysis/lm_negative.csv', 'r') as f:
    reader = csv.reader(f)
    for row in reader:
        entry = row[0].strip().split(" ")
        if len(entry) > 1:
            negative.extend(entry)
        else:
            negative.append(entry[0])

final_lex = {}
final_lex.update({word:2.0 for word in positive})
final_lex.update({word:-2.0 for word in negative})
final_lex.update(stock_lex_scaled)
final_lex.update(sia.lexicon)
sia.lexicon = final_lex

We use polarity_scores method of SentimentIntensityAnalyzer from VaderSentiment api to calculate the sentiment scores(positive,negative and neutral).

In [2]:
# Calculating sentiment scores 
def get_sentiment_scores(sentence):
    snt = sia.polarity_scores(sentence)
    return format(str(snt))

#### Extract the cleaned data and calculate sentiment scores

We apply html_regex and regex to clean the required data from 10-Q and 10-K sec files and calculate the sentiment scores on them. 

In [4]:
# Function for extracting requried text
def rawdata_extract(path, processingFile, cik, ticker, fdate, form):
    html_regex = re.compile(r'<.*?>')
    extraxted_data=[]
    
    filenameopen = os.path.join(path, processingFile)
    print("taking this: " + filenameopen)
    resultdict = dict()
    resultdict['CIK'] = cik
    resultdict['CONAME'] = ticker
    resultdict['FDATE'] = fdate
    resultdict['FORM'] = form
    resultdict['SECFNAME'] = processingFile
                
    with open(filenameopen, 'r' , errors="replace") as in_file:
        content = in_file.read().lower()
        content = re.sub(html_regex,'',content) 
        content = content.replace('&nbsp;','') 
        content = re.sub(r'&#\d+;', '', content)
        content = re.sub(r'[^\x00-\x7f]', ' ', content)
        content = re.sub(' +',' ', content)
        if content:
            content = str(content).replace('\n', '')
              #create tokens 
            tokens = word_tokenize(content)
            text = nltk.Text(tokens)  
            total_tokens = len(tokens)
            total_itr = int(total_tokens/2000)
    
            #spliting file into batches of 2000 words and calculating sentiment scores
            limit = 0
            end_limit = 2000
            negscore_array = np.array([])
            neuscore_array = np.array([])
            posscore_array = np.array([])
            
            for i in range(total_itr):
                first_list = tokens[limit:limit+2000]
                first_list_str = " ".join(first_list)
                #caclulate sentiment scores for 2000 words per iteration
                content_scores = get_sentiment_scores(first_list_str)
                limit = limit + 2000  

                #convert each score to numpy array
                negscores = float(content_scores.split(",")[0].split(":")[1])
                neuscores = float(content_scores.split(",")[1].split(":")[1])
                posscores = float(content_scores.split(",")[2].split(":")[1])

                negscore_array = np.append(negscore_array, negscores)
                neuscore_array = np.append(neuscore_array, neuscores)
                posscore_array = np.append(posscore_array, posscores)   

            neg_meanscore = np.mean(negscore_array)
            neu_meanscore = np.mean(neuscore_array)
            pos_meanscore = np.mean(posscore_array)

            resultdict['neg_score'] = neg_meanscore
            resultdict['neu_score'] = neu_meanscore
            resultdict['pos_score'] = pos_meanscore
            
            print("scores calculated")
            print(resultdict)
            exit
        extraxted_data.append(resultdict)
        in_file.close()
    return extraxted_data

##### Get the corresponding CIKs for the 100 selected tickers and form a ciklist that is used for sentiment analysis.

In [6]:
## filter the ciks which we need for sentiment analysis
cik_ticker_path = common_path +'/analysisEdgar/Edgar_filter_cik_ticker_map.csv'
cik_tickerdf = pd.read_csv(cik_ticker_path)
cik_tickernewdf = cik_tickerdf[['cik','ticker']]

tickers = ['MMM', 'ABT', 'ABMD', 'ACN', 'ATVI', 'ADBE', 'AMD', 'AAP', 'AES', 'AMG', 'AFL', 'A', 'APD', 'AKAM', 'ALK', 'ALB', 'ARE', 'ALXN', 'ALGN', 'AGN', 'ADS', 'LNT', 'ALL', 'GOOGL', 'MO', 'AMZN', 'AEE', 'AAL', 'AEP', 'AXP', 'AIG', 'AMT', 'AMP', 'ABC', 'AME', 'AMGN', 'APH', 'APC', 'ADI', 'ANSS', 'ANTM', 'AON', 'AOS', 'APA', 'AIV', 'AAPL', 'AMAT', 'ADM', 'ARNC', 'AJG', 'AIZ', 'ATO', 'T', 'ADSK', 'ADP', 'AZO', 'AVB', 'AVY', 'BLL', 'BAC', 'BK', 'BAX', 'BBT', 'BDX', 'BBY', 'BIIB', 'BLK', 'HRB', 'BA', 'BWA', 'BXP', 'BSX', 'BMY', 'BR', 'CHRW', 'COG', 'CDNS', 'CPB', 'COF', 'CAH', 'KMX', 'CCL', 'CAT', 'CBS', 'CE', 'CELG', 'CNC', 'CNP', 'CTL', 'CERN', 'CF', 'SCHW', 'CVX', 'CMG', 'CB', 'CHD', 'CI', 'XEC', 'CINF', 'CTAS', 'CSCO', 'C', 'CTXS', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'CL', 'CMCSA', 'CMA', 'CAG', 'CXO', 'COP', 'ED', 'STZ', 'COO', 'CPRT', 'GLW', 'COST', 'CCI', 'CSX', 'CMI', 'CVS', 'DHI', 'DHR', 'DRI', 'DVA', 'DE', 'DAL', 'XRAY', 'DVN', 'DLR', 'DFS', 'DISCA', 'DISH', 'DLTR', 'D', 'DOV', 'DTE', 'DRE', 'DUK', 'ETFC', 'EMN', 'ETN', 'EBAY', 'ECL', 'EIX', 'EW', 'EA', 'EMR', 'ETR', 'EOG', 'EFX', 'EQIX', 'EQR', 'ESS', 'EL', 'ES', 'RE', 'EXC', 'EXPE', 'EXPD', 'EXR', 'XOM', 'FFIV', 'FAST', 'FRT', 'FDX', 'FIS', 'FITB', 'FE', 'FISV', 'FLIR', 'FLS', 'FLR', 'FMC', 'FL', 'F', 'BEN', 'FCX', 'GPS', 'GRMN', 'IT', 'GD', 'GE', 'GIS', 'GPC', 'GILD', 'GPN', 'GS', 'GWW', 'HAL', 'HBI', 'HOG', 'HRS', 'HIG', 'HAS', 'HCP', 'HP', 'HSIC', 'HSY', 'HES', 'HFC', 'HOLX', 'HD', 'HON', 'HRL', 'HST', 'HPQ', 'HUM', 'HBAN', 'IDXX', 'ITW', 'ILMN', 'IR', 'INTC', 'ICE', 'IBM', 'INCY', 'IP', 'IPG', 'IFF', 'INTU', 'ISRG', 'IVZ', 'IPGP', 'IRM', 'JKHY', 'JEC', 'JBHT', 'SJM', 'JNJ', 'JCI', 'JPM', 'JNPR', 'KSU', 'K', 'KEY', 'KMB', 'KIM', 'KLAC', 'KSS', 'KR', 'LB', 'LLL', 'LH', 'LRCX', 'LEG', 'LEN', 'LLY', 'LNC', 'LKQ', 'LMT', 'L', 'LOW', 'MTB', 'MAC', 'M', 'MRO', 'MAR', 'MMC', 'MLM', 'MAS', 'MA', 'MAT', 'MKC', 'MXIM', 'MCD', 'MCK', 'MDT', 'MRK', 'MET', 'MTD', 'MGM', 'MCHP', 'MU', 'MSFT', 'MAA', 'MHK', 'TAP', 'MDLZ', 'MNST', 'MCO', 'MS', 'MOS', 'MSI', 'MSCI', 'MYL', 'NDAQ', 'NOV', 'NKTR', 'NTAP', 'NFLX', 'NWL', 'NEM', 'NEE', 'NKE', 'NI', 'NBL', 'JWN', 'NSC', 'NTRS', 'NOC', 'NRG', 'NUE', 'NVDA', 'ORLY', 'OXY', 'OMC', 'OKE', 'ORCL', 'PCAR', 'PKG', 'PH', 'PAYX', 'PNR', 'PBCT', 'PEP', 'PKI', 'PRGO', 'PFE', 'PNW', 'PXD', 'PNC', 'RL', 'PPL', 'PFG', 'PG', 'PGR', 'PLD', 'PRU', 'PEG', 'PSA', 'PHM', 'PVH', 'PWR', 'QCOM', 'DGX', 'RJF', 'RTN', 'O', 'RHT', 'REG', 'REGN', 'RF', 'RSG', 'RMD', 'RHI', 'ROK', 'ROL', 'ROP', 'ROST', 'RCL', 'CRM', 'SBAC', 'SLB', 'STX', 'SEE', 'SRE', 'SHW', 'SPG', 'SWKS', 'SLG', 'SNA', 'SO', 'LUV', 'SWK', 'SBUX', 'STT', 'SYK', 'STI', 'SIVB', 'SYMC', 'SNPS', 'SYY', 'TROW', 'TTWO', 'TGT', 'TEL', 'FTI', 'TFX', 'TXN', 'TXT', 'TMO', 'TIF', 'TJX', 'TMK', 'TSS', 'TSCO', 'TDG', 'FOXA', 'FOX', 'TSN', 'UDR', 'ULTA', 'USB', 'UAA', 'UNP', 'UAL', 'UNH', 'UPS', 'URI', 'UTX', 'UHS', 'UNM', 'VFC', 'VLO', 'VAR', 'VTR', 'VRSN', 'VZ', 'VRTX', 'VIAB', 'VNO', 'VMC', 'WAB', 'WMT', 'WBA', 'DIS', 'WM', 'WAT', 'WEC', 'WCG', 'WFC', 'WDC', 'WU', 'WY', 'WHR', 'WMB', 'WYNN', 'XEL', 'XRX', 'XLNX', 'YUM', 'ZBH', 'ZION']
tickers = [x.lower() for x in tickers]
ticker_df = pd.DataFrame(tickers, columns=['ticker'])

join_df = pd.merge(cik_tickernewdf, ticker_df, on='ticker')
cik_list = join_df["cik"].astype(str).tolist()

##### Calculate the sentiment scores for 10-K sec files of 100 ciks for the past 10 years and save the scores in ticker csv file.

In [24]:
import ast

path = common_path+'/data/scraped_10k'
cikListFile = common_path+'/data/ciklist/10-K'
sent_scorefile = common_path+'/SEC_Sentiments/10-K'

os.chdir(cikListFile)
file_list = [fname for fname in os.listdir()]

for files in file_list:   
    ciks = files.split('.')[0]
    if ciks in cik_list:
        cikListFile = pd.read_csv(files)
        for index, row in cikListFile.iterrows():
            processingFile=row['filename'] 
            cik=row['cik'] 
            ticker=row['ticker']  
            fdate = row['Date'] 
            form = row['sec_type'] 

            cik_name = row['filename'].split("_")
            file_path = path + '/'+cik_name[0]+'/clean_data'

            datalist = rawdata_extract(file_path, processingFile, cik, ticker, fdate, form)
            result_df = pd.DataFrame(datalist)
#             # headers are CIK	CONAME	FDATE	FORM	SECFNAME	neg_score	neu_score	pos_score
            result_df.to_csv(sent_scorefile + '/' + cik_name[0]+'.csv',encoding='utf-8', mode = 'a', index=False, header=False)
            print("file written")

taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001120193/clean_data/0001120193_2013-02-21.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001120193/clean_data/0001120193_2014-02-24.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001120193/clean_data/0001120193_2019-02-22.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001120193/clean_data/0001120193_2012-02-24.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001120193/clean_data/0001120193_2008-02-25.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001120193/clean_data/0001120193_2011-02-24.txt
scores calc

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000027419/clean_data/0000027419_2017-03-08.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000027419/clean_data/0000027419_2019-03-13.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000027419/clean_data/0000027419_2012-03-15.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000027419/clean_data/0000027419_2018-03-14.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000027419/clean_data/0000027419_2015-03-13.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000027419/clean_data/000002

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004977/clean_data/0000004977_2019-02-25.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004977/clean_data/0000004977_2012-02-27.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004977/clean_data/0000004977_2015-02-26.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004977/clean_data/0000004977_2018-02-23.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004977/clean_data/0000004977_2013-02-27.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004977/clean_data/000000

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004962/clean_data/0000004962_2012-02-24.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004962/clean_data/0000004962_2017-02-17.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004962/clean_data/0000004962_2015-02-24.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004962/clean_data/0000004962_2013-02-22.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004962/clean_data/0000004962_2016-02-19.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004962/clean_data/000000

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001002910/clean_data/0001002910_2008-02-29.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001002910/clean_data/0001002910_2013-03-01.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001002910/clean_data/0001002910_2011-02-24.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001002910/clean_data/0001002910_2009-03-02.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001002910/clean_data/0001002910_2012-02-28.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001002910/clean_data/000100

...

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001045609/clean_data/0001045609_2017-02-15.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001045609/clean_data/0001045609_2008-02-29.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0001045609/clean_data/0001045609_2014-02-26.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000072903/clean_data/0000072903_2016-02-19.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000072903/clean_data/0000072903_2009-02-27.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000072903/clean_data/000007

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000783280/clean_data/0000783280_2011-02-25.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000783280/clean_data/0000783280_2010-03-01.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000100885/clean_data/0000100885_2008-02-28.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000100885/clean_data/0000100885_2011-02-04.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000100885/clean_data/0000100885_2017-02-03.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000100885/clean_data/000010

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000874761/clean_data/0000874761_2011-02-28.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004281/clean_data/0000004281_2019-02-21.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004281/clean_data/0000004281_2017-02-28.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004281/clean_data/0000004281_2011-02-17.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004281/clean_data/0000004281_2008-02-15.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000004281/clean_data/000000

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000936340/clean_data/0000936340_2019-02-07.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000936340/clean_data/0000936340_2012-02-16.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000936340/clean_data/0000936340_2011-02-18.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000936340/clean_data/0000936340_2017-02-09.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000936340/clean_data/0000936340_2014-02-14.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10k/0000936340/clean_data/000093

KeyboardInterrupt: 

##### Calculate the sentiment scores for 10-Q sec files of 100 ciks for the past 10 years and save the scores in ticker csv file.

In [84]:
import ast

path = common_path+'/data/scraped_10q'
cikListFile = common_path+'/data/ciklist/10-Q'
sent_scorefile = common_path+'/SEC_Sentiments/10-Q'

os.chdir(cikListFile)
file_list = [fname for fname in os.listdir()]

#converting the cik column back to string
tenk_ciks_list = tenk_ciks_df['cik'].astype('str').tolist()

for files in file_list:
    ciks = files.split('.')[0]
    if ciks in tenk_ciks_list:        
        cikListFile = pd.read_csv(files)
        for index, row in cikListFile.iterrows():
            processingFile=row['filename'] 
            cik=row['cik'] 
            ticker=row['ticker']  
            fdate = row['Date'] 
            form = row['sec_type'] 

            cik_name = row['filename'].split("_")
            file_path = path + '/'+cik_name[0]+'/clean_data'

            datalist = rawdata_extract(file_path, processingFile, cik, ticker, fdate, form)
            result_df = pd.DataFrame(datalist)
            # headers are CIK	CONAME	FDATE	FORM	SECFNAME	neg_score	neu_score	pos_score
            result_df.to_csv(sent_scorefile + '/' + cik_name[0]+'.csv',encoding='utf-8', mode = 'a', index=False, header=False)
            print("file written")

taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0001120193/clean_data/0001120193_2014-11-05.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0001120193/clean_data/0001120193_2015-05-06.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0001120193/clean_data/0001120193_2009-05-08.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0001120193/clean_data/0001120193_2018-05-02.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0001120193/clean_data/0001120193_2017-08-02.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0001120193/clean_data/0001120193_2013-05-07.txt
scores calc

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000006769/clean_data/0000006769_2013-08-06.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000006769/clean_data/0000006769_2018-08-02.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000006769/clean_data/0000006769_2014-05-09.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000006769/clean_data/0000006769_2009-11-06.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000006769/clean_data/0000006769_2015-08-07.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000006769/clean_data/000000

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0001156039/clean_data/0001156039_2013-07-24.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000865752/clean_data/0000865752_2016-11-07.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000865752/clean_data/0000865752_2010-08-09.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000865752/clean_data/0000865752_2015-08-10.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000865752/clean_data/0000865752_2008-05-12.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000865752/clean_data/000086

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000027419/clean_data/0000027419_2009-06-05.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000027419/clean_data/0000027419_2009-08-28.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000027419/clean_data/0000027419_2018-08-27.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000027419/clean_data/0000027419_2015-08-25.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000027419/clean_data/0000027419_2008-12-05.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000027419/clean_data/000002

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0001408198/clean_data/0001408198_2017-05-05.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0001408198/clean_data/0001408198_2010-04-08.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000882095/clean_data/0000882095_2009-11-05.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000882095/clean_data/0000882095_2008-10-31.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000882095/clean_data/0000882095_2017-05-10.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000882095/clean_data/000088

...

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000062709/clean_data/0000062709_2011-08-05.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000062709/clean_data/0000062709_2017-04-28.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000062709/clean_data/0000062709_2014-11-04.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000062709/clean_data/0000062709_2018-10-26.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000062709/clean_data/0000062709_2012-08-08.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000062709/clean_data/000006

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000077476/clean_data/0000077476_2016-04-18.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000077476/clean_data/0000077476_2012-07-25.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000077476/clean_data/0000077476_2014-07-23.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000077476/clean_data/0000077476_2009-04-22.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000077476/clean_data/0000077476_2012-10-17.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000077476/clean_data/000007

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000936340/clean_data/0000936340_2016-04-26.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000936340/clean_data/0000936340_2017-10-25.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000936340/clean_data/0000936340_2012-07-30.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000936340/clean_data/0000936340_2011-07-29.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000936340/clean_data/0000936340_2008-05-12.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000936340/clean_data/000093

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000100517/clean_data/0000100517_2011-07-26.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000100517/clean_data/0000100517_2018-04-18.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000100517/clean_data/0000100517_2008-07-24.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000100517/clean_data/0000100517_2008-05-09.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000100517/clean_data/0000100517_2013-04-25.txt
scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0000100517/clean_data/000010

scores calculated
file written
taking this: /Users/kiranrawat/Desktop/portfolio-risk-assessment-using-ml/data/scraped_10q/0001306830/clean_data/0001306830_2013-04-19.txt
scores calculated
file written


##### Generate the 100 tickers list as a csv file as we need them for stock price prediction part.

In [9]:
# generate list of tickers for which we have 10-K sentiment scores
tenk_path = common_path+'/SEC_Sentiments/10-K'

os.chdir(tenk_path)
tenk_file_list = [fname for fname in os.listdir()]
tenk_file_ciks = [fname.split('.')[0] for fname in tenk_file_list]
tenk_file_ciks = list(filter(None, tenk_file_ciks))

#joining two dataframes to generate the tickers which are present in both dataframes
tenk_ciks_df = pd.DataFrame(tenk_file_ciks, columns=['cik']).astype('int64')
final_100_ciks = pd.merge(join_df, tenk_ciks_df, on='cik')
final_100_tickers = final_100_ciks['ticker']
final_100_tickers.to_csv(common_path+'/tickers_100.csv',encoding='utf-8', header=True)